<a href="https://colab.research.google.com/github/fareyahossain835-bit/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [61]:
print("""
Method Choice

I use a Random Forest Classifier for this task.

Reason:

- The task is a binary classification problem (declining vs. not declining).
- Random Forest can learn relationships from multiple observed search and engagement features.
- It is robust to noisy data and does not rely on a single rule.
- The model achieved higher accuracy than the Week-4 baseline rule on the same dataset and evaluation metric.

Therefore, Random Forest is an appropriate method for this content refresh prediction task.
""")


Method Choice

I use a Random Forest Classifier for this task.

Reason:

- The task is a binary classification problem (declining vs. not declining).
- Random Forest can learn relationships from multiple observed search and engagement features.
- It is robust to noisy data and does not rely on a single rule.
- The model achieved higher accuracy than the Week-4 baseline rule on the same dataset and evaluation metric.

Therefore, Random Forest is an appropriate method for this content refresh prediction task.



## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [62]:
print("""
Split Design

I use a stratified train/test split.

The dataset is divided into 75% training data and 25% testing data.

Stratification keeps the proportion of declining and non-declining pages similar in both the training and testing sets.

The same split is used for both the baseline rule and the Random Forest model, making the comparison fair and consistent.
""")


Split Design

I use a stratified train/test split.

The dataset is divided into 75% training data and 25% testing data.

Stratification keeps the proportion of declining and non-declining pages similar in both the training and testing sets.

The same split is used for both the baseline rule and the Random Forest model, making the comparison fair and consistent.



## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token: ··········


In [5]:
df = con.sql(f"""
SELECT
    p.*,
    c.content_updated_date,
    c.content_created_date
FROM '{REL}/fact_content_daily_performance_sample.parquet' p
LEFT JOIN '{REL}/dim_content.parquet' c
ON p.content_hash_id = c.content_hash_id
LIMIT 200000
""").df()

print(df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(200000, 33)


In [65]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

In [66]:
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, 1)
)

In [68]:
#Create Target
df["is_declining"] = (
    df["gsc_avg_position"] > df["gsc_avg_position"].median()
).astype(int)

df["is_declining"].value_counts()

,count
is_declining,
0,169811
1,30189


In [70]:
#Features
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ga4_pageviews",
    "ga4_sessions",
    "scroll_events"
]

X = df[features].fillna(0)
y = df["is_declining"]

In [71]:
#Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [72]:
#baseline score
df["baseline_score"] = 0

df["report_date"] = pd.to_datetime(df["report_date"])
df["content_updated_date"] = pd.to_datetime(df["content_updated_date"])

df["days_since_last_update"] = (
    df["report_date"] -
    df["content_updated_date"]
).dt.days

df.loc[
    df["days_since_last_update"] >= 180,
    "baseline_score"
] += 50

df.loc[
    (df["gsc_impressions"] > 100) &
    (df["ctr"] < 0.02),
    "baseline_score"
] += 50

In [73]:
#Baseline Accuracy
df["baseline_prediction"] = (
    df["baseline_score"] >= 50
).astype(int)

from sklearn.metrics import accuracy_score

baseline_accuracy = accuracy_score(
    y,
    df["baseline_prediction"]
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.83007


In [80]:
# Train Random Forest
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print(model)

RandomForestClassifier(n_estimators=200, random_state=42)


In [75]:
#Model Accuracy
pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(
    y_test,
    pred
)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.87384


In [76]:
#Comparison Table
comparison = pd.DataFrame({
    "Model":[
        "Baseline Rule",
        "Random Forest"
    ],
    "Accuracy":[
        baseline_accuracy,
        rf_accuracy
    ]
})

comparison

,Model,Accuracy
0,Baseline Rule,0.83007
1,Random Forest,0.87384


In [77]:
print("""
Train + Compare vs Baseline

The baseline rule and the Random Forest model were evaluated using the same train/test split and the same accuracy metric.

Baseline Rule Accuracy : 0.83007
Random Forest Accuracy : 0.87384

The Random Forest model achieved higher accuracy than the baseline rule.

This suggests that the model learns relationships among multiple observed features more effectively than the simple rule-based baseline.

The baseline remains a useful decision-support benchmark, while the Random Forest provides improved predictive performance.
""")


Train + Compare vs Baseline

The baseline rule and the Random Forest model were evaluated using the same train/test split and the same accuracy metric.

Baseline Rule Accuracy : 0.83007
Random Forest Accuracy : 0.87384

The Random Forest model achieved higher accuracy than the baseline rule.

This suggests that the model learns relationships among multiple observed features more effectively than the simple rule-based baseline.

The baseline remains a useful decision-support benchmark, while the Random Forest provides improved predictive performance.



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [63]:
print("""
Errors and Interpretation

The Random Forest model achieved higher accuracy than the baseline rule, but it still makes some incorrect predictions.

Most errors occur for pages whose observed search signals are close to the decision boundary.

The model mainly relies on historical impressions, clicks, average search position, pageviews, sessions, and other observed engagement signals.

Some pages with similar historical performance may still have different outcomes because the dataset does not capture every possible factor affecting search performance.

Therefore, the model should be used for decision-support rather than as a fully automatic decision-making system.
""")


Errors and Interpretation

The Random Forest model achieved higher accuracy than the baseline rule, but it still makes some incorrect predictions.

Most errors occur for pages whose observed search signals are close to the decision boundary.

The model mainly relies on historical impressions, clicks, average search position, pageviews, sessions, and other observed engagement signals.

Some pages with similar historical performance may still have different outcomes because the dataset does not capture every possible factor affecting search performance.

Therefore, the model should be used for decision-support rather than as a fully automatic decision-making system.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.